In [ ]:
import glob, os, warnings
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
warnings.filterwarnings("ignore")

# ----------------------------------------------------------------- load data
def find(patterns):
    for root in ["/kaggle/input", "."]:
        for p in patterns:
            hits = sorted(glob.glob(os.path.join(root, "**", p), recursive=True))
            if hits:
                return hits[0]
    raise FileNotFoundError(patterns)

train = pd.read_csv(find(["train.csv"]))
test = pd.read_csv(find(["test.csv", "test*.csv"]))
sample = pd.read_csv(find(["sample_submission.csv", "submission.csv"]))

# ------------------------------------------------------- feature engineering
def make_features(d):
    d = d.copy()
    d["route"] = d.source_city + ">" + d.destination_city
    d["stops_n"] = d.stops.map({"zero": 0, "one": 1, "two_or_more": 2})
    d["log_dur"] = np.log(d.duration)
    d["log_days"] = np.log(d.days_left)
    d["cls_flight"] = d.flight_class + "_" + d.flight
    d["cls_air_route"] = d.flight_class + "_" + d.airline + "_" + d.route
    return d

X, X_test = make_features(train), make_features(test)
y = train["price"].values

num_cols = ["duration", "days_left", "stops_n", "log_dur", "log_days"]
small_cats = ["airline", "departure_time", "arrival_time", "flight_class", "route", "stops"]
big_cats = ["flight", "cls_flight", "cls_air_route"]

# ------------------------------------------------------------- model pipeline
def build_model():
    pre = ColumnTransformer([
        # (1) polynomial expansion of numeric features (degree tuned below)
        ("num_poly", Pipeline([
            ("scale", StandardScaler()),
            ("poly", PolynomialFeatures(degree=3, include_bias=False)),
            ("scale2", StandardScaler()),
        ]), num_cols),
        # (2) one-hot categoricals
        ("cat", OneHotEncoder(handle_unknown="ignore"), small_cats + big_cats),
        # (3) degree-2 interactions between categorical dummies and numerics
        ("cross", Pipeline([
            ("enc", ColumnTransformer([
                ("c", OneHotEncoder(handle_unknown="ignore", sparse_output=False), small_cats),
                ("n", StandardScaler(), num_cols),
            ])),
            ("poly", PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
        ]), small_cats + num_cols),
    ], sparse_threshold=0.3)
    ridge = Pipeline([("pre", pre), ("ridge", Ridge())])
    # log-transformed target: fit on log1p(price), predict expm1(...)
    return TransformedTargetRegressor(regressor=ridge, func=np.log1p, inverse_func=np.expm1)

# ------------------------------------------- hyperparameter tuning, 5-fold CV
param_grid = {
    "regressor__pre__num_poly__poly__degree": [2, 3],
    "regressor__ridge__alpha": [1, 3, 5, 10],
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(build_model(), param_grid, cv=cv, scoring="r2", n_jobs=1, verbose=1)
grid.fit(X, y)
print("Best params :", grid.best_params_)
print(f"Best CV R2  : {grid.best_score_:.4f}")

oof = cross_val_predict(grid.best_estimator_, X, y, cv=cv)
print(f"OOF R2 / RMSE: {r2_score(y, oof):.4f} / {np.sqrt(np.mean((y - oof) ** 2)):.2f}")

# --------------------------------------------------------- predict + submit
pred = np.clip(grid.best_estimator_.predict(X_test), 0, None)
sub = pd.DataFrame({sample.columns[0]: test["id"].values,
                    sample.columns[1]: pred})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("submission shape:", sub.shape, "| columns:", list(sub.columns))